In [4]:
import ee
import numpy as np
import rasterio
import glob
from tqdm import tqdm
from multiprocessing.dummy import Lock, Pool
from datetime import date, datetime, timedelta
from random import randint
import urllib
import os
import re
import time

In [5]:
ee.Authenticate()

True

In [6]:
ee.Initialize()

In [7]:
class Counter:
    def __init__(self, start: int = 0) -> None:
        self.value = start
        self.lock = Lock()

    def update(self, delta: int = 1) -> int:
        with self.lock:
            self.value += delta
            return self.value

In [8]:
def date2str(date: datetime) -> str:
    return date.strftime("%Y-%m-%d")

def get_period(date: datetime, days: int = 15) -> tuple[str, str]:
    date1 = date - timedelta(days=days / 2)
    date2 = date + timedelta(days=days / 2)
    return (
        date2str(date1),
        date2str(date2),
    )

In [9]:
s2_root = 'EuroSATallBands/ds/images/remote_sensing/otherDatasets/sentinel_2/tif'
s2_fnames = glob.glob(s2_root+'/*/*.tif')

start_time = time.time()
counter = Counter()
startdate = date.fromisoformat('2016-06-01')

In [10]:
def get_url(crs_str,bbox_list):
    region = ee.Geometry.Rectangle(coords=bbox_list,proj=crs_str,geodesic=True,evenOdd=False)
    collection = ee.ImageCollection('COPERNICUS/S1_GRD')
    date=startdate + timedelta(randint(1,365))
    date1,date2 = get_period(date, 15)
    collection = collection.filterDate(date1,date2)
    collection = collection.filterBounds(region)
    try:
        image = collection.first().select('VV','VH')            
        download_params = {'region':region.getInfo(),'format':'GEO_TIFF'}
        img_url = image.toFloat().getDownloadURL(download_params)    
    except:
        img_url = get_url(crs_str,bbox_list)
    return img_url

In [11]:
def worker(idx):

    pattern = r'(.*/)(.*)/(.*)'
    match = re.match(pattern, s2_fnames[idx])
    save_dir = 'EuroSAT-SAR-expand/'+ match.group(2)
    os.makedirs(save_dir, exist_ok=True)    
    save_path = save_dir + '/' + match.group(3)
    if os.path.exists(save_path):
        count = counter.update(1)

    else:

        with rasterio.open(s2_fnames[idx],'r') as rf:
            crs = rf.crs
            bounds = rf.bounds
        
        crs_str = crs.to_string()
        
        # Get bounding box in [left, bottom, right, top] format
        bbox_list = [bounds.left, bounds.bottom, bounds.right, bounds.top]

        # Expand bounding box to double the edge length (64x64 -> 128x128)
        center_x = (bounds.left + bounds.right) / 2
        center_y = (bounds.bottom + bounds.top) / 2
        width = bounds.right - bounds.left
        height = bounds.top - bounds.bottom
        new_width = 2 * width
        new_height = 2 * height
        new_left = center_x - new_width / 2
        new_right = center_x + new_width / 2
        new_bottom = center_y - new_height / 2
        new_top = center_y + new_height / 2
        bbox_list = [new_left, new_bottom, new_right, new_top]

        img_url = get_url(crs_str,bbox_list)
        urllib.request.urlretrieve(img_url, save_path)
        count = counter.update(1)
        
    if count % 100 == 0:
        print(f"Downloaded {count} images in {time.time() - start_time:.3f}s.")

In [12]:
indices = np.arange(len(s2_fnames))
with Pool(processes=8) as p:
    p.map(worker, indices)

Downloaded 100 images in 33.454s.


KeyboardInterrupt: 

Downloaded 200 images in 59.171s.
Downloaded 300 images in 84.591s.
Downloaded 400 images in 109.237s.
Downloaded 500 images in 131.890s.
Downloaded 600 images in 153.354s.
Downloaded 700 images in 175.274s.
Downloaded 800 images in 196.277s.
Downloaded 900 images in 218.141s.
Downloaded 1000 images in 239.504s.
Downloaded 1100 images in 260.182s.
Downloaded 1200 images in 281.614s.
Downloaded 1300 images in 304.162s.
Downloaded 1400 images in 326.649s.
Downloaded 1500 images in 346.220s.
Downloaded 1600 images in 367.161s.
Downloaded 1700 images in 385.903s.
Downloaded 1800 images in 405.521s.
Downloaded 1900 images in 424.914s.
Downloaded 2000 images in 443.940s.
Downloaded 2100 images in 462.764s.
Downloaded 2200 images in 480.694s.
Downloaded 2300 images in 498.680s.
Downloaded 2400 images in 515.813s.
Downloaded 2500 images in 535.173s.
Downloaded 2600 images in 552.674s.
Downloaded 2700 images in 570.583s.
Downloaded 2800 images in 588.517s.
Downloaded 2900 images in 606.968s.
Do